
# 5. Statistics and Campaign Analysis

## 5.1 Campaign Data Exploration

In this notebook, we will analyze marketing campaigns and coupon redemption behavior.

We will use Python, MySQL, SciPy, and statsmodels.

Our main goals are:

- Understand campaign participation.
- Calculate coupon redemption rates.
- Compare customer purchasing behavior.
- Apply statistical hypothesis tests.
- Interpret statistical and business significance.

We will begin with descriptive analysis before conducting statistical tests.

We will not assume that differences between customer groups are caused by marketing campaigns.

In [17]:
from pathlib import Path
import sys
from sqlalchemy import text
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency, mannwhitneyu
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.database import get_engine

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

engine = get_engine()

In [3]:
# Load small campaign tables
queries = {
    "campaign_desc": "SELECT * FROM raw_campaign_desc",
    "campaign_table": "SELECT * FROM raw_campaign_table",
    "coupon_redempt": "SELECT * FROM raw_coupon_redempt",
}

campaign_data = {}

with engine.connect() as connection:
    for name, query in queries.items():

        campaign_data[name] = pd.read_sql_query(
            text(query),
            con=connection
        )

# Store DataFrames
campaign_desc = campaign_data["campaign_desc"]
campaign_table = campaign_data["campaign_table"]
coupon_redempt = campaign_data["coupon_redempt"]

# Display dataset information
for name, df in campaign_data.items():
    print(f"{name}: {df.shape}")
    print("Columns:", df.columns.tolist())
    print()

display(campaign_desc.head())

campaign_desc: (30, 4)
Columns: ['CAMPAIGN', 'DESCRIPTION', 'START_DAY', 'END_DAY']

campaign_table: (7208, 3)
Columns: ['DESCRIPTION', 'household_key', 'CAMPAIGN']

coupon_redempt: (2318, 4)
Columns: ['household_key', 'DAY', 'COUPON_UPC', 'CAMPAIGN']



,CAMPAIGN,DESCRIPTION,START_DAY,END_DAY
0,1,TypeB,346,383
1,2,TypeB,351,383
2,3,TypeC,356,412
3,4,TypeB,372,404
4,5,TypeB,377,411


In [4]:
# Count targeted households by campaign
campaign_targets = (
    campaign_table
    .groupby("CAMPAIGN", as_index=False)
    .agg(
        targeted_households=("household_key", "nunique")
    )
)

# Count coupon redemptions by campaign
campaign_redemptions = (
    coupon_redempt
    .groupby("CAMPAIGN", as_index=False)
    .agg(
        redemption_events=("household_key", "size"),
        redeeming_households=("household_key", "nunique"),
    )
)

# Combine campaign information
campaign_overview = (
    campaign_desc
    .merge(
        campaign_targets,
        on="CAMPAIGN",
        how="left",
        validate="one_to_one",
    )
    .merge(
        campaign_redemptions,
        on="CAMPAIGN",
        how="left",
        validate="one_to_one",
    )
)

# Fill missing counts
count_cols = [
    "targeted_households",
    "redemption_events",
    "redeeming_households",
]

campaign_overview[count_cols] = (
    campaign_overview[count_cols].fillna(0).astype(int)
)

# Calculate household redemption rate
campaign_overview["redemption_rate_pct"] = (
    campaign_overview["redeeming_households"]
    / campaign_overview["targeted_households"].replace(0,np.nan)
    * 100
)

# Sort campaigns
campaign_overview = campaign_overview.sort_values(
    "redemption_rate_pct",
    ascending=False,
)

display(campaign_overview.head(10).round(2))

,CAMPAIGN,DESCRIPTION,START_DAY,END_DAY,targeted_households,redemption_events,redeeming_households,redemption_rate_pct
17,18,TypeA,587,642,1133,653,214,18.89
12,13,TypeA,504,551,1077,629,196,18.20
2,3,TypeC,356,412,12,2,2,16.67
7,8,TypeA,412,460,1076,372,158,14.68
24,25,TypeB,659,691,187,61,24,12.83
22,23,TypeB,646,684,183,60,23,12.57
14,15,TypeC,547,708,17,2,2,11.76
18,19,TypeB,603,635,130,29,15,11.54
8,9,TypeB,435,467,176,43,20,11.36
28,29,TypeB,281,334,118,24,13,11.02



## 5.3 Campaign Type Comparison

We will compare coupon redemption behavior across campaign types.

Our dataset contains three campaign categories:

- TypeA
- TypeB
- TypeC

We will calculate redemption rates using household-campaign assignments.

A household can participate in more than one campaign.

We will also compare the number of campaigns and unique targeted households.

Our goal is to understand whether redemption behavior differs across campaign types.

These results describe associations, not causal campaign effects.


In [5]:
campaign_table.head()

,DESCRIPTION,household_key,CAMPAIGN
0,TypeA,1,8
1,TypeB,1,12
2,TypeA,1,13
3,TypeA,1,18
4,TypeC,1,20


In [7]:
# 1. Select unique campaign assignments
campaign_targets_df = campaign_table[
    ["CAMPAIGN", "household_key"]
].drop_duplicates()

# 2. Identify households that redeemed coupons
redemption_flags = (
    coupon_redempt[
        ["CAMPAIGN", "household_key"]
    ]
    .drop_duplicates()
    .assign(redeemed=1)
)

# 3. Validate campaign assignments
assignment_check = redemption_flags.merge(
    campaign_targets_df,
    on=["CAMPAIGN", "household_key"],
    how="left",
    indicator=True,
)

unmatched = (
    assignment_check["_merge"] == "left_only"
).sum()

print("Unmatched Redemptions:", unmatched)

assert unmatched == 0

# 4. Merge campaign information
campaign_participation = (
    campaign_targets_df
    .merge(
        campaign_desc[
            ["CAMPAIGN", "DESCRIPTION"]
        ],
        on="CAMPAIGN",
        how="left",
        validate="many_to_one",
    )
    .merge(
        redemption_flags,
        on=["CAMPAIGN", "household_key"],
        how="left",
        validate="one_to_one",
    )
)

# 5. Create binary redemption indicator
campaign_participation["redeemed"] = (
    campaign_participation["redeemed"]
    .fillna(0)
    .astype(int)
)

# 6. Validate the dataset
assert campaign_participation["DESCRIPTION"].notna().all()
assert campaign_participation["redeemed"].isin([0, 1]).all()

print("Participation Dataset Shape:", campaign_participation.shape)

display(campaign_participation.head())




Unmatched Redemptions: 0
Participation Dataset Shape: (7208, 4)


,CAMPAIGN,household_key,DESCRIPTION,redeemed
0,8,1,TypeA,1
1,12,1,TypeB,0
2,13,1,TypeA,0
3,18,1,TypeA,1
4,20,1,TypeC,0


In [8]:
# Aggregate household-campaign assignments by type
campaign_type_summary = (
    campaign_participation
    .groupby("DESCRIPTION", as_index=False)
    .agg(
        campaign_count=("CAMPAIGN", "nunique"),
        targeted_assignments=("household_key", "size"),
        unique_households=("household_key", "nunique"),
        redeemed_assignments=("redeemed", "sum"),
    )
)

# Calculate redemption rate
campaign_type_summary["redemption_rate_pct"] = (
    campaign_type_summary["redeemed_assignments"]
    / campaign_type_summary["targeted_assignments"]
    * 100
)

# Sort by redemption rate
campaign_type_summary = (
    campaign_type_summary
    .sort_values(
        "redemption_rate_pct",
        ascending=False
    )
)

display(campaign_type_summary.round(2))

,DESCRIPTION,campaign_count,targeted_assignments,unique_households,redeemed_assignments,redemption_rate_pct
0,TypeA,5,3979,1513,635,15.96
1,TypeB,19,2655,1023,210,7.91
2,TypeC,6,574,397,44,7.67



## 5.4 Chi-Square Test of Independence

We will use the Chi-Square Test to investigate the relationship between campaign type and coupon redemption.

Our hypotheses are:

- H0: Campaign type and coupon redemption are independent.
- H1: Campaign type and coupon redemption are associated.

The significance level is 0.05.

We will create a contingency table and calculate the Chi-Square statistic, p-value, and Cramer's V.

Because some households participate in multiple campaigns, the independence assumption is not fully satisfied.

Therefore, this test is exploratory. We will not use it as causal evidence.


In [10]:
# 1. Create the contingency table
contingency_table = pd.crosstab(
    campaign_participation["DESCRIPTION"],
    campaign_participation["redeemed"],
)

contingency_table = contingency_table.reindex(
    columns=[0, 1],
    fill_value=0,
)

print("CONTINGENCY TABLE")
display(
    contingency_table.rename(
        columns={0: "Not Redeemed", 1: "Redeemed"}
    )
)

# 2. Perform Chi-Square Test
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

# 3. Calculate Cramer's V
n = contingency_table.to_numpy().sum()

min_dimension = min(
    contingency_table.shape[0] - 1,
    contingency_table.shape[1] - 1,
)

cramers_v = np.sqrt(
    chi2 / (n * min_dimension)
)

# 4. Display test results
print("\nCHI-SQUARE TEST RESULTS")
print(f"Chi-Square Statistic: {chi2:.4f}")
print(f"P-Value: {p_value:.6e}")
print(f"Degrees of Freedom: {dof}")
print(f"Minimum Expected Frequency: {expected.min():.2f}")
print(f"Cramer's V: {cramers_v:.4f}")

# 5. Display expected frequencies
expected_df = pd.DataFrame(
    expected,
    index=contingency_table.index,
    columns=["Not Redeemed", "Redeemed"],
)

print("\nEXPECTED FREQUENCIES")
display(expected_df.round(2))

CONTINGENCY TABLE


redeemed,Not Redeemed,Redeemed
DESCRIPTION,,
TypeA,3344,635
TypeB,2445,210
TypeC,530,44



CHI-SQUARE TEST RESULTS
Chi-Square Statistic: 107.9902
P-Value: 3.549997e-24
Degrees of Freedom: 2
Minimum Expected Frequency: 70.79
Cramer's V: 0.1224

EXPECTED FREQUENCIES


,Not Redeemed,Redeemed
DESCRIPTION,,
TypeA,"3,488.25",490.75
TypeB,"2,327.55",327.45
TypeC,503.21,70.79



## 5.5 GEE Analysis

We will use a GEE logistic model to analyze coupon redemption.

The model accounts for repeated observations from the same household.

TypeB will be our reference campaign type.

We will examine odds ratios, confidence intervals, and p-values.


In [12]:
# Müşteri bazında tekrarlanan gözlemleri dikkate al
gee_model = sm.GEE.from_formula(
    formula='redeemed ~ C(DESCRIPTION, Treatment(reference="TypeB"))',
    groups="household_key",
    data=campaign_participation,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable(),
)

gee_result = gee_model.fit()

# Odds Ratio ve güven aralıkları
gee_summary = pd.DataFrame({
    "odds_ratio": np.exp(gee_result.params),
    "ci_lower": np.exp(gee_result.conf_int()[0]),
    "ci_upper": np.exp(gee_result.conf_int()[1]),
    "p_value": gee_result.pvalues,
})

display(gee_summary.round(4))

,odds_ratio,ci_lower,ci_upper,p_value
Intercept,0.06,0.05,0.07,0.00
"C(DESCRIPTION, Treatment(reference=""TypeB""))[T.TypeA]",3.20,2.55,4.01,0.00
"C(DESCRIPTION, Treatment(reference=""TypeB""))[T.TypeC]",0.92,0.54,1.56,0.75



## 5.6 Campaign Spending Analysis

We will compare spending behavior between households that redeemed coupons and households that did not.

We will calculate campaign-period spending for each household-campaign assignment.

Our goals are to:

- Compare average and median spending.
- Examine spending differences across campaign types.
- Prepare data for statistical testing.

These comparisons show associations, not causal campaign effects.


In [13]:
campaign_participation.head()

,CAMPAIGN,household_key,DESCRIPTION,redeemed
0,8,1,TypeA,1
1,12,1,TypeB,0
2,13,1,TypeA,0
3,18,1,TypeA,1
4,20,1,TypeC,0


In [15]:
query = text("""
    WITH basket_totals AS (
        SELECT
            household_key,
            BASKET_ID,
            MIN(`DAY`) AS basket_day,
            SUM(SALES_VALUE) AS basket_sales
        FROM raw_transaction_data
        GROUP BY household_key, BASKET_ID
        HAVING SUM(SALES_VALUE) > 0
    )
    SELECT
        ct.CAMPAIGN,
        ct.household_key,
        COUNT(bt.BASKET_ID) AS campaign_baskets,
        COALESCE(SUM(bt.basket_sales), 0) AS campaign_spending
    FROM (
        SELECT DISTINCT CAMPAIGN, household_key
        FROM raw_campaign_table
    ) AS ct
    JOIN raw_campaign_desc AS cd
        ON ct.CAMPAIGN = cd.CAMPAIGN
    LEFT JOIN basket_totals AS bt
        ON ct.household_key = bt.household_key
        AND bt.basket_day BETWEEN cd.START_DAY AND cd.END_DAY
    GROUP BY ct.CAMPAIGN, ct.household_key
""")

with engine.connect() as connection:
    campaign_sales = pd.read_sql_query(
        query,
        con=connection
    )

# Convert monetary values
campaign_sales["campaign_spending"] = pd.to_numeric(
    campaign_sales["campaign_spending"]
)

# Combine spending and coupon redemption information
campaign_spending_df = campaign_participation.merge(
    campaign_sales,
    on=["CAMPAIGN", "household_key"],
    how="left",
    validate="one_to_one"
)

# Validate data
assert len(campaign_spending_df) == len(campaign_participation)
assert campaign_spending_df["campaign_spending"].notna().all()
assert campaign_spending_df["campaign_spending"].ge(0).all()

print("Campaign Spending Dataset:", campaign_spending_df.shape)

display(campaign_spending_df.head())

Campaign Spending Dataset: (7208, 6)


,CAMPAIGN,household_key,DESCRIPTION,redeemed,campaign_baskets,campaign_spending
0,8,1,TypeA,1,7,429.01
1,12,1,TypeB,0,6,205.41
2,13,1,TypeA,0,8,209.97
3,18,1,TypeA,1,9,465.67
4,20,1,TypeC,0,8,426.44


In [16]:
# Compare spending by redemption status
spending_summary = (
    campaign_spending_df
    .groupby(["DESCRIPTION", "redeemed"])
    .agg(
        assignments=("household_key", "size"),
        avg_spending=("campaign_spending", "mean"),
        median_spending=("campaign_spending", "median"),
        avg_baskets=("campaign_baskets", "mean"),
    )
    .reset_index()
)

display(spending_summary.round(2))

,DESCRIPTION,redeemed,assignments,avg_spending,median_spending,avg_baskets
0,TypeA,0,3344,362.04,279.71,12.64
1,TypeA,1,635,625.14,540.29,16.43
2,TypeB,0,2445,402.18,329.29,11.66
3,TypeB,1,210,518.12,462.56,13.47
4,TypeC,0,530,765.03,612.52,21.31
5,TypeC,1,44,"1,095.40","1,015.70",26.09



## 5.8 Mann-Whitney U Test

We will compare spending distributions between households that redeemed coupons and those that did not.

We will use the Mann-Whitney U test because spending values may be skewed.

We will perform separate tests for each campaign type and adjust p-values for multiple comparisons.

These tests are exploratory because some households participate in multiple campaigns.


In [18]:
test_results = []

for campaign_type, group in campaign_spending_df.groupby("DESCRIPTION"):

    # Customers who use coupons
    redeemed = group.loc[
        group["redeemed"] == 1,
        "campaign_spending"
    ].to_numpy(dtype=float)

    # Customers who do not use coupons
    not_redeemed = group.loc[
        group["redeemed"] == 0,
        "campaign_spending"
    ].to_numpy(dtype=float)

    # Mann-Whitney U Test
    statistic, p_value = mannwhitneyu(
        redeemed,
        not_redeemed,
        alternative="two-sided",
        method="asymptotic"
    )

    # Cliff's Delta: effect size
    cliffs_delta = (
        2 * statistic / (len(redeemed) * len(not_redeemed))
        -1
    )

    test_results.append({
        "campaign_type": campaign_type,
        "redeemed_n": len(redeemed),
        "not_redeemed_n": len(not_redeemed),
        "median_redeemed": np.median(redeemed),
        "median_not_redeemed": np.median(not_redeemed),
        "u_statistic": statistic,
        "p_value": p_value,
        "cliffs_delta": cliffs_delta,
    })

mann_whitney_results = pd.DataFrame(test_results)

# Multiple testing correction (Holm)
mann_whitney_results["p_adjusted"] = multipletests(
    mann_whitney_results["p_value"],
    method="holm"
)[1]

display(mann_whitney_results.round(4))

,campaign_type,redeemed_n,not_redeemed_n,median_redeemed,median_not_redeemed,u_statistic,p_value,cliffs_delta,p_adjusted
0,TypeA,635,3344,540.29,279.71,"1,555,371.00",0.00,0.47,0.00
1,TypeB,210,2445,462.56,329.29,"324,280.00",0.00,0.26,0.00
2,TypeC,44,530,"1,015.71",612.52,"16,042.00",0.00,0.38,0.00



## 5.9 GEE Spending Analysis

We will use a GEE model to analyze campaign spending.

The dependent variable is log-transformed campaign spending.

The model includes coupon redemption, campaign type, their interaction, and campaign duration.

Household ID will be used to account for repeated observations.

This analysis measures associations, not causal effects.


In [19]:
# Add campaign dates
gee_spending_df = campaign_spending_df.merge(
    campaign_desc[
        ["CAMPAIGN", "START_DAY", "END_DAY"]
    ],
    on="CAMPAIGN",
    how="left",
    validate="many_to_one",
)

# Calculate campaign duration
gee_spending_df["campaign_duration"] = (
    gee_spending_df["END_DAY"]
    - gee_spending_df["START_DAY"]
    + 1
)

# Transform spending
gee_spending_df["log_spending"] = np.log1p(
    gee_spending_df["campaign_spending"]
)

# Fit GEE model
gee_spending_model = sm.GEE.from_formula(
    formula=(
        'log_spending ~ redeemed * '
        'C(DESCRIPTION, Treatment(reference="TypeB")) '
        '+ campaign_duration'
    ),
    groups="household_key",
    data=gee_spending_df,
    family=sm.families.Gaussian(),
    cov_struct=sm.cov_struct.Exchangeable(),
)

gee_spending_result = gee_spending_model.fit()

# Extract coefficients and confidence intervals
gee_spending_summary = pd.DataFrame({
    "coefficient": gee_spending_result.params,
    "ci_lower": gee_spending_result.conf_int()[0],
    "ci_upper": gee_spending_result.conf_int()[1],
    "p_value": gee_spending_result.pvalues,
})

display(gee_spending_summary.round(4))

,coefficient,ci_lower,ci_upper,p_value
Intercept,4.43,4.32,4.54,0.00
"C(DESCRIPTION, Treatment(reference=""TypeB""))[T.TypeA]",0.10,0.04,0.16,0.00
"C(DESCRIPTION, Treatment(reference=""TypeB""))[T.TypeC]",0.23,0.15,0.32,0.00
redeemed,0.29,0.19,0.39,0.00
"redeemed:C(DESCRIPTION, Treatment(reference=""TypeB""))[T.TypeA]",0.03,-0.07,0.13,0.54
"redeemed:C(DESCRIPTION, Treatment(reference=""TypeB""))[T.TypeC]",-0.04,-0.21,0.14,0.68
campaign_duration,0.01,0.01,0.02,0.00



## 5.10 Key Findings and Conclusions

### Campaign Redemption

- TypeA achieved the highest redemption rate (15.96%).
- TypeB and TypeC achieved similar redemption rates.
- The Chi-Square test found a significant association between campaign type and coupon redemption.
- GEE analysis showed that TypeA had higher redemption odds than TypeB.

### Campaign Spending

- Customers who redeemed coupons had higher observed spending.
- Mann-Whitney U tests found significant differences across all three campaign types.
- GEE analysis found a positive association between coupon redemption and spending.
- Redemption interaction effects were not statistically significant.

### Limitations

- The same household can participate in multiple campaigns.
- Customers were not randomly assigned to redemption groups.
- Campaign timing and customer selection may influence the results.
- Observed spending differences do not represent incremental revenue.

### Conclusion

Campaign type and coupon redemption are associated with customer purchasing behavior.

TypeA campaigns showed stronger redemption performance.

Coupon redemption was also associated with higher spending.

However, causal analysis is required to estimate the true business impact of marketing campaigns.
